In [2]:
from google.colab import files
uploaded = files.upload()

Saving motion_test.mp4 to motion_test.mp4


In [3]:
import cv2
import numpy as np
from moviepy.editor import VideoFileClip

input_video = 'motion_test.mp4'
output_video = 'security_alert.mp4'

# Step 1: Create VideoCapture and VideoWriter objects.
cap = cv2.VideoCapture(input_video)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

# Step 2: Create the background subtractor.
bg_subtractor = cv2.createBackgroundSubtractorKNN(history=500)

# Step 3: Define the erosion kernel.
kernel = np.ones((5, 5), np.uint8)

frame_count = 0

# Step 4: Write the processing loop.
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # a) Apply background subtraction
    fg_mask = bg_subtractor.apply(frame)

    # b) Erode the foreground mask
    fg_mask_eroded = cv2.erode(fg_mask, kernel, iterations=1)

    # c) Use findNonZero() to check for motion
    motion_points = cv2.findNonZero(fg_mask_eroded)

    if motion_points is not None:
        # d) If motion is detected
        x, y, w, h = cv2.boundingRect(motion_points)

        # Draw bounding box in red
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 0, 255), 2)

        # Add red banner
        cv2.rectangle(frame, (0, 0), (width, 50), (0, 0, 255), -1)
        cv2.putText(frame,
                    f'ALERT: Motion Detected - Frame: {frame_count}',
                    (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    2)
    else:
        # e) If NO motion
        cv2.rectangle(frame, (0, 0), (width, 50), (0, 255, 0), -1)
        cv2.putText(frame,
                    f'Status: Clear - Frame: {frame_count}',
                    (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (255, 255, 255),
                    2)

    # f) Write the frame
    out.write(frame)

# Step 5: Release all objects.
cap.release()
out.release()
cv2.destroyAllWindows()

# Step 6: Preview the output.
clip = VideoFileClip(output_video)
clip.ipython_display(width=800)

Moviepy - Building video __temp__.mp4.
Moviepy - Writing video __temp__.mp4



Moviepy - Done !
Moviepy - video ready __temp__.mp4
